# Catalog Verification - Basic Statistics

Perform some basic verification on the datasets.

- confirm the number of nulls (NaNs) in the dataset is within expectations
- for fields with predictable limits, confirm min/max values in dataset

In [ ]:
import os
import hats
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from hats.io.validation import is_valid_catalog, is_valid_collection
from pathlib import Path

In [ ]:
VERSION = os.environ["VERSION"]
OUTPUT_DIR = Path(os.environ["OUTPUT_DIR"])

print(f"VERSION: {VERSION}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

hats_dir = OUTPUT_DIR / "hats" / VERSION

## Convenience methods

We define a few convenience methods to load catalog metadata, check for known value types and their ranges, and output the number of nulls found in each column.

This ensures we're performing the same kinds of checks against each table type.

In [ ]:
def min_max_floats_in_range(stats, columns, exp_min_value, exp_max_value, value_type):
    """Check all columns (rows) of a value type against `min_value`/`max_value` in the stats
    dataframe. Columns absent from the stats are reported, not fatal."""
    if not columns:
        return
    missing = [c for c in columns if c not in stats.index]
    if missing:
        print(f"**** {value_type} columns missing from statistics: {missing}")
    present = [c for c in columns if c in stats.index]
    anything_bad = False
    for col in present:
        if float(stats["min_value"][col]) < exp_min_value or float(stats["max_value"][col]) > exp_max_value:
            print(
                f"**** {col} has values outside acceptable range for {value_type} "
                f"(min:{stats['min_value'][col]}, max:{stats['max_value'][col]})"
            )
            anything_bad = True
    if not anything_bad and present:
        print(f"  All {value_type} columns within acceptable ranges ({present})")


def nested_leaf_consistency(catalog_path, nested_columns):
    """Guard against broken nested columns: within every partition file, all leaf columns of a
    nested (struct-of-lists) column must carry the same number of values. Reads only
    dataset/_metadata. (This is the check that catches offsets corruption — see notebook 11.)"""
    md_file = Path(catalog_path) / "dataset" / "_metadata"
    if not md_file.exists():
        print("  nested consistency: no _metadata — run the notebook 11 footer scan instead")
        return
    md = pq.read_metadata(md_file)
    counts = {}  # (file, nested_col) -> {field: num_values}
    for rg_i in range(md.num_row_groups):
        rg = md.row_group(rg_i)
        fp = rg.column(0).file_path
        for c_i in range(rg.num_columns):
            col = rg.column(c_i)
            parts = col.path_in_schema.split(".")
            if parts[0] in nested_columns and len(parts) > 1:
                per = counts.setdefault((fp, parts[0]), {})
                per[parts[1]] = per.get(parts[1], 0) + col.num_values
    bad = {k: v for k, v in counts.items() if len(set(v.values())) > 1}
    if bad:
        for (fp, ncol), fields in bad.items():
            deviant = {f: n for f, n in fields.items() if n != max(set(fields.values()), key=list(fields.values()).count)}
            print(f"**** nested column {ncol!r} INCONSISTENT in {fp}: {deviant}")
    else:
        print(f"  nested consistency: OK for {nested_columns} across {len({k[0] for k in counts})} files")


def verify_catalog(
    catalog_name,
    ra_cols=None,
    dec_cols=None,
    flux_cols=None,
    flux_err_cols=None,
    mjd_cols=None,
    nested_columns=None,
    base_path=None,
):
    print(catalog_name)
    path = (base_path or hats_dir) / catalog_name
    if not path.exists():
        print("  SKIP — not found:", path)
        return
    cat = hats.read_hats(path)
    print("  is valid catalog", is_valid_catalog(path, strict=True, verbose=False))
    print("  num partitions:", len(cat.get_healpix_pixels()))
    print("  num rows:", cat.catalog_info.total_rows)
    stats = cat.aggregate_column_statistics()
    print("  num columns:", len(stats))
    ## Remove columns with "Mag" as these are created for HATS
    stats = stats.iloc[~stats.index.str.contains("Mag")]

    min_max_floats_in_range(stats, ra_cols, 0.0, 360.0, "RIGHT ASCENSION")
    min_max_floats_in_range(stats, dec_cols, -90.0, 90.0, "DECLINATION")
    min_max_floats_in_range(stats, flux_cols, -100_000_000.0, 100_000_000.0, "FLUX")
    min_max_floats_in_range(stats, flux_err_cols, 0.0, 100_000_000.0, "FLUX ERROR")
    min_max_floats_in_range(stats, mjd_cols, 60600.0, 60700.0, "MJD")

    if nested_columns:
        nested_leaf_consistency(path, nested_columns)

    np_null_count = stats["null_count"].to_numpy(copy=False, dtype=np.int64)
    with_null_index = np.where(np_null_count > 0)
    with_nulls = stats.iloc[with_null_index]

    if len(with_nulls):
        print(f"  columns with nulls: {len(with_nulls)}")
        with_nulls = with_nulls[["null_count"]]
        with_nulls["percent"] = [
            null_count / cat.catalog_info.total_rows * 100
            for null_count in with_nulls["null_count"].to_numpy(copy=False, dtype=np.int64)
        ]
        with_nulls = with_nulls.sort_values(by="percent", ascending=False)
        print(with_nulls)
    else:
        print("  columns with nulls: 0")


def verify_collection(collection_name, main_catalog, nested_columns, **verify_kwargs):
    """Verify a HATS collection: the main nested catalog (+ column checks), its margin
    cache(s), and its index catalog(s)."""
    path = hats_dir / collection_name
    print(f"===== {collection_name} =====")
    if not path.exists():
        print("  SKIP — not found:", path)
        return
    print("  is valid collection", is_valid_collection(path))

    verify_catalog(main_catalog, base_path=path, nested_columns=nested_columns, **verify_kwargs)

    main_rows = hats.read_hats(path / main_catalog).catalog_info.total_rows
    others = sorted(
        d for d in path.iterdir() if d.is_dir() and d.name != main_catalog and (d / "dataset").is_dir()
    )
    for d in others:
        kind = "margin" if d.name.endswith("arcs") else "index"
        valid = is_valid_catalog(d, strict=True, verbose=False)
        rows = hats.read_hats(d).catalog_info.total_rows
        note = ""
        if kind == "index" and rows != main_rows:
            note = f"  **** expected {main_rows:,} rows (one per main-catalog row)"
        print(f"  {kind} {d.name}: valid={valid}, rows={rows:,}{note}")

In [ ]:
verify_catalog("dia_object", ra_cols=["ra"], dec_cols=["dec"], mjd_cols=[])

In [ ]:
verify_catalog(
    "dia_source",
    ra_cols=["ra", "trailRa"],
    dec_cols=["dec", "trailDec"],
    flux_cols=[
        "apFlux",
        "psfFlux",
        "trailFlux",
        "dipoleMeanFlux",
        "dipoleFluxDiff",
        "scienceFlux",
        "ixxPSF",
        "iyyPSF",
        "ixyPSF",
    ],
    flux_err_cols=[
        "apFluxErr",
        "psfFluxErr",
        "dipoleMeanFluxErr",
        "dipoleFluxDiffErr",
        "scienceFluxErr",
    ],
    mjd_cols=["midpointMjdTai"],
)

In [ ]:
verify_catalog(
    "dia_object_forced_source",
    ra_cols=["coord_ra"],
    dec_cols=["coord_dec"],
    flux_cols=["psfFlux", "psfDiffFlux"],
    flux_err_cols=["psfFluxErr", "psfDiffFluxErr"],
    mjd_cols=["midpointMjdTai"],
)

In [ ]:
verify_catalog(
    "object",
    ra_cols=["coord_ra"],
    dec_cols=["coord_dec"],
    flux_cols=[
        "u_psfFlux",
        "u_kronFlux",
        "g_psfFlux",
        "g_kronFlux",
        "r_psfFlux",
        "r_kronFlux",
        "i_psfFlux",
        "i_kronFlux",
        "z_psfFlux",
        "z_kronFlux",
        "y_psfFlux",
        "y_kronFlux",
    ],
    flux_err_cols=[
        "u_psfFluxErr",
        "u_kronFluxErr",
        "g_psfFluxErr",
        "g_kronFluxErr",
        "r_psfFluxErr",
        "r_kronFluxErr",
        "i_psfFluxErr",
        "i_kronFluxErr",
        "z_psfFluxErr",
        "z_kronFluxErr",
        "y_psfFluxErr",
        "y_kronFluxErr",
    ],
)

In [ ]:
# pd.set_option('display.max_rows', None)

verify_catalog(
    "source",
    ra_cols=["ra"],
    dec_cols=["dec"],
    flux_cols=[
        "calibFlux",
        "ap03Flux",
        "ap06Flux",
        "ap09Flux",
        "ap12Flux",
        "ap17Flux",
        "ap25Flux",
        "ap35Flux",
        "ap50Flux",
        "ap70Flux",
        "psfFlux",
        "gaussianFlux",
        "apFlux_12_0_instFlux",
        "apFlux_17_0_instFlux",
        "apFlux_35_0_instFlux",
        "apFlux_50_0_instFlux",
        "normCompTophatFlux_instFlux",
        "localBackground_instFlux",
    ],
    flux_err_cols=[
        "calibFluxErr",
        "ap03FluxErr",
        "ap06FluxErr",
        "ap09FluxErr",
        "ap12FluxErr",
        "ap17FluxErr",
        "ap25FluxErr",
        "ap35FluxErr",
        "ap50FluxErr",
        "ap70FluxErr",
        "psfFluxErr",
        "gaussianFluxErr",
        "apFlux_12_0_instFluxErr",
        "apFlux_17_0_instFluxErr",
        "apFlux_35_0_instFluxErr",
        "apFlux_50_0_instFluxErr",
        "normCompTophatFlux_instFluxErr",
        "localBackground_instFluxErr",
    ],
    mjd_cols=["midpointMjdTai"],
)

In [ ]:
verify_catalog(
    "object_forced_source",
    ra_cols=["coord_ra"],
    dec_cols=["coord_dec"],
    flux_cols=["psfFlux", "psfDiffFlux"],
    flux_err_cols=["psfFluxErr", "psfDiffFluxErr"],
    mjd_cols=["midpointMjdTai"],
)

## Collections

The pipeline's collections stage moves each nested light-curve catalog into a collection
directory alongside its margin cache and object-id index (`collections` section of the
config). Verify the whole bundle: the main nested catalog (including range checks on the
nested `<column>.<field>` leaves and the nested-offsets consistency guard from notebook
11), the margin, and the index.

In [ ]:
DIA_SOURCE_FLUX = ['apFlux', 'psfFlux', 'trailFlux', 'dipoleMeanFlux', 'dipoleFluxDiff', 'scienceFlux', 'ixxPSF', 'iyyPSF', 'ixyPSF']
DIA_SOURCE_FLUXERR = ['apFluxErr', 'psfFluxErr', 'dipoleMeanFluxErr', 'dipoleFluxDiffErr', 'scienceFluxErr']
FORCED_FLUX = ['psfFlux', 'psfDiffFlux']
FORCED_FLUXERR = ['psfFluxErr', 'psfDiffFluxErr']
OBJ_FLUX = ['u_psfFlux', 'u_kronFlux', 'g_psfFlux', 'g_kronFlux', 'r_psfFlux', 'r_kronFlux', 'i_psfFlux', 'i_kronFlux', 'z_psfFlux', 'z_kronFlux', 'y_psfFlux', 'y_kronFlux']
OBJ_FLUXERR = ['u_psfFluxErr', 'u_kronFluxErr', 'g_psfFluxErr', 'g_kronFluxErr', 'r_psfFluxErr', 'r_kronFluxErr', 'i_psfFluxErr', 'i_kronFluxErr', 'z_psfFluxErr', 'z_kronFluxErr', 'y_psfFluxErr', 'y_kronFluxErr']


def pref(prefix, cols):
    return [f"{prefix}.{c}" for c in cols]

In [ ]:
verify_collection(
    "dia_object_collection",
    main_catalog="dia_object_lc",
    nested_columns=["diaSource", "diaObjectForcedSource"],
    ra_cols=["ra"] + pref("diaSource", ["ra", "trailRa"]) + ["diaObjectForcedSource.coord_ra"],
    dec_cols=["dec"] + pref("diaSource", ["dec", "trailDec"]) + ["diaObjectForcedSource.coord_dec"],
    flux_cols=pref("diaSource", DIA_SOURCE_FLUX) + pref("diaObjectForcedSource", FORCED_FLUX),
    flux_err_cols=pref("diaSource", DIA_SOURCE_FLUXERR) + pref("diaObjectForcedSource", FORCED_FLUXERR),
    mjd_cols=["diaSource.midpointMjdTai", "diaObjectForcedSource.midpointMjdTai"],
)

In [ ]:
verify_collection(
    "object_collection",
    main_catalog="object_lc",
    nested_columns=["objectForcedSource"],
    ra_cols=["coord_ra", "objectForcedSource.coord_ra"],
    dec_cols=["coord_dec", "objectForcedSource.coord_dec"],
    flux_cols=OBJ_FLUX + pref("objectForcedSource", FORCED_FLUX),
    flux_err_cols=OBJ_FLUXERR + pref("objectForcedSource", FORCED_FLUXERR),
    mjd_cols=["objectForcedSource.midpointMjdTai"],
)

## Uncertainty-corrected collections

The uncertainty-correction stage writes a collection per input collection, named
`<collection>_uncertainty_corrected`, with the main catalog keeping its original name
(`dia_object_lc` / `object_lc`) and the margin carried through the correction. The
nested forced-source column gains `*_corrected` and `*_corrected_flag` columns; corrected
errors must obey the same flux-error range as the raw ones (flags are booleans, not
range-checked here).

In [ ]:
verify_collection(
    "dia_object_collection_uncertainty_corrected",
    main_catalog="dia_object_lc",
    nested_columns=["diaSource", "diaObjectForcedSource"],
    flux_err_cols=pref("diaObjectForcedSource", FORCED_FLUXERR)
    + pref("diaObjectForcedSource", ["psfFluxErr_corrected", "psfDiffFluxErr_corrected"]),
    mjd_cols=["diaObjectForcedSource.midpointMjdTai"],
)

In [ ]:
verify_collection(
    "object_collection_uncertainty_corrected",
    main_catalog="object_lc",
    nested_columns=["objectForcedSource"],
    flux_err_cols=pref("objectForcedSource", FORCED_FLUXERR)
    + pref("objectForcedSource", ["psfFluxErr_corrected", "psfDiffFluxErr_corrected"]),
    mjd_cols=["objectForcedSource.midpointMjdTai"],
)